# caption-single-revisar.ipynb — ouvir cada divergência e decidir

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

O `caption-single-generate` lista onde o Whisper e o roteiro discordam. Este
notebook faz a parte que sobra: **toca o trecho do áudio** de cada divergência,
com as duas versões do texto lado a lado, pra você decidir ouvindo.

## Por que ouvir, e não escolher no papel

O SRT tem duas coisas e só uma pode estar errada:

| | Vem de | Confiável? |
|---|---|---|
| tempos | Whisper ouvindo o áudio | sim |
| texto | Whisper adivinhando grafia | erra nome próprio |

O roteiro tem o texto certo e nenhum tempo. Mas **o David Williams lê
ligeiramente diferente do escrito em alguns trechos** — 0,9625 medido no
`biblia-audio-conferir`. Nesses pontos o Whisper está certo e o roteiro não:
a legenda tem que dizer o que se ouve.

Só o áudio separa "o Whisper errou" de "o Dave leu diferente". Este notebook
existe pra essa separação levar dois minutos em vez de uma tarde.

## A regra padrão, quando você não quer ouvir

> **Faz sentido em inglês?** Sim → fica o Whisper. Não → vai pro roteiro.

Ela acerta o caso comum (o Whisper erra grafia de nome próprio, e grafia
errada não é palavra) sem exigir que você ouça nada. Mas é **julgamento**, não
regra que o código aplica sozinho: "Seeing their treasures" é gramatical e
mesmo assim está errado. Por isso este notebook mostra, e você decide.

⚠️ **Nada é escrito aqui.** No fim ele imprime as trocas que você marcou, pra
você aplicar no SRT baixado.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — Drive e módulos (rode uma vez por sessão)             ║
# ╚══════════════════════════════════════════════════════════════════╝

!apt-get -qq -y install ffmpeg > /dev/null 2>&1
print('✅ ffmpeg')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if not PASTA_MODULOS.exists():
    raise SystemExit(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos")

# ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
# "N módulos" sozinho não quer dizer nada. E o modo de falhar aqui é
# traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
# então um copytree logo depois do mount às vezes enxerga só parte dos
# arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde.
_manifesto = PASTA_MODULOS / "_manifesto.txt"
if not _manifesto.exists():
    print("   ⚠️  sem _manifesto.txt no Drive — ele é versionado no repositório,")
    print("      então rode o repositorio-sincronizar pra trazê-lo")
else:
    _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                  if l.strip() and not l.startswith("#")}
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO_MODULOS.glob("*.py")}
    _fora_do_drive = sorted(_esperados - _no_drive)
    _nao_copiados  = sorted((_esperados & _no_drive) - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO_MODULOS / _n)
        _na_vm = {f.name for f in DESTINO_MODULOS.glob("*.py")}
        _nao_copiados = sorted((_esperados & _no_drive) - _na_vm)
    if _fora_do_drive:
        print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
        for _n in _fora_do_drive:
            print(f"     {_n}")
        raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_esperados)} módulos do manifesto estão na VM")

if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

NOME_ORACAO = "40_Matt_02"
IDIOMA_MESTRE = "en"

# Segundos de folga antes e depois do trecho. Um pouco de contexto ajuda o
# ouvido a situar a frase; muito faz você ouvir o capítulo inteiro.
FOLGA_SEG = 1.5

print(f"Capítulo ... {NOME_ORACAO}")
print(f"Folga ...... {FOLGA_SEG}s antes e depois")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 CARREGAR — áudio, SRT e roteiro                               ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import audio_narracao, biblia_texto as bt
from config import PipelineConfig
from srt_utils import ler_srt

config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
                        IDIOMA_MESTRE=IDIOMA_MESTRE)

# ── áudio ────────────────────────────────────────────────────────────────
AUDIO = Path(config.NOME_AUDIO)
origem = audio_narracao.trazer(config, AUDIO)
if not AUDIO.exists():
    raise SystemExit(audio_narracao.erro_nao_achei(config))
print(f"🔊 {AUDIO.name} — {origem}")

# ── SRT ──────────────────────────────────────────────────────────────────
SRT = Path(config.NOME_SRT_PT_WHISPER)
if not SRT.exists():
    fonte = config.pasta_oracao / SRT.name
    if not fonte.exists():
        raise SystemExit(f"❌ {SRT.name} não está em {config.pasta_oracao}. "
                         f"Rode o caption-single-generate primeiro.")
    shutil.copyfile(fonte, SRT)
legendas = ler_srt(SRT)
print(f"📝 {SRT.name} — {len(legendas)} blocos")

# ── roteiro ──────────────────────────────────────────────────────────────
roteiro_txt = config.pasta_oracao / f"{NOME_ORACAO}_roteiro_versiculos.txt"
if roteiro_txt.exists():
    referencia, origem_ref = roteiro_txt.read_text(encoding="utf-8"), roteiro_txt.name
elif config.caminho_web_biblia.exists():
    referencia = bt.roteiro_do_capitulo(NOME_ORACAO, config.caminho_web_biblia)
    origem_ref = "web-biblia.json"
else:
    raise SystemExit("❌ Sem roteiro pra comparar. Rode o biblia-texto-baixar uma vez.")
print(f"📄 {origem_ref}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 ALINHAR — onde os dois discordam, e em que segundo             ║
# ╚══════════════════════════════════════════════════════════════════╝

texto_whisper = " ".join(l.texto for l in legendas)
cmp = bt.comparar(texto_whisper, referencia)

print(f"similaridade {cmp.similaridade:.4f} · "
      f"Whisper {cmp.palavras_a} palavras · roteiro {cmp.palavras_b}")

if cmp.identico:
    print("\n✅ Idênticos — nada a revisar.")
    divergencias = []
else:
    # Do índice da palavra pro bloco, e do bloco pro tempo. É por isso que a
    # posição vem do alinhamento (cmp.posicoes_a) e não de procurar o trecho
    # no texto: buscar pela primeira palavra do contexto casaria a primeira
    # ocorrência dela no capítulo, e o áudio tocado seria o do lugar errado.
    limites, acc = [], 0
    for i, l in enumerate(legendas):
        acc += len(bt.palavras_comparaveis(l.texto))
        limites.append((acc, i))

    def bloco_de(pos):
        for limite, i in limites:
            if pos < limite:
                return i
        return len(legendas) - 1

    divergencias = []
    for (tipo, do_whisper, do_roteiro), pos in zip(cmp.diferencas, cmp.posicoes_a):
        i = bloco_de(pos)
        divergencias.append({"n": len(divergencias) + 1, "bloco": i + 1, "tipo": tipo,
                             "whisper": do_whisper, "roteiro": do_roteiro,
                             "inicio": legendas[i].inicio_seg, "fim": legendas[i].fim_seg})

    print(f"\n⚠️  {len(divergencias)} trecho(s) a revisar")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎧 OUVIR — um player por divergência                             ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from IPython.display import Audio, display, HTML

Path("trechos").mkdir(exist_ok=True)

for d in divergencias:
    ini = max(0.0, d["inicio"] - FOLGA_SEG)
    dur = (d["fim"] - d["inicio"]) + 2 * FOLGA_SEG
    corte = Path("trechos") / f"div_{d['n']:02d}.wav"
    if not corte.exists():
        subprocess.run(["ffmpeg", "-y", "-ss", str(ini), "-t", str(dur),
                        "-i", str(AUDIO), str(corte)], capture_output=True)

    display(HTML(
        f"<div style='margin:14px 0;padding:10px;border-left:3px solid #888'>"
        f"<b>#{d['n']} · bloco {d['bloco']} · {d['inicio']:.1f}s</b><br>"
        f"<span style='color:#06c'>Whisper:</span> …{d['whisper']}…<br>"
        f"<span style='color:#c60'>Roteiro:</span> …{d['roteiro']}…"
        f"</div>"))
    display(Audio(str(corte)))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✍️  DECIDIR — marque o que vai pro roteiro                       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Ponha aqui os números (#) das divergências em que o ROTEIRO ganha. As que
# não estiverem na lista ficam como o Whisper transcreveu.
#
# A regra padrão, quando você não quiser ouvir:
#     faz sentido em inglês?  sim → Whisper (não põe na lista)
#                             não → roteiro (põe na lista)
#
# Ela não é automática de propósito: "Seeing their treasures" é gramatical e
# mesmo assim está errado. Julgamento é seu; o notebook só mostra.

VAO_PRO_ROTEIRO = []      # ex: [3, 7, 11]

if not VAO_PRO_ROTEIRO:
    print("Nenhuma marcada — tudo fica como o Whisper transcreveu.")
else:
    print(f"📋 {len(VAO_PRO_ROTEIRO)} troca(s) a fazer no SRT, à mão:\n")
    for n in VAO_PRO_ROTEIRO:
        d = next((x for x in divergencias if x["n"] == n), None)
        if d is None:
            print(f"   ⚠️  #{n} não existe na lista (são 1..{len(divergencias)})")
            continue
        print(f"   bloco {d['bloco']}  ({d['inicio']:.1f}s)")
        print(f"      de:   …{d['whisper']}…")
        print(f"      para: …{d['roteiro']}…")
        print()
    print("Abra o SRT baixado, faça essas trocas e resuba pro Drive com o")
    print("mesmo nome. O caption-single-burn lê o que estiver lá na hora.")

# Este notebook NÃO escreve no SRT de propósito. O contexto impresso é o
# trecho NORMALIZADO (sem pontuação, minúsculo) -- serve pra você achar o
# lugar, não pra colar. Trocar automático colaria texto sem pontuação no
# meio da legenda, e isso sairia no vídeo.